## Imports

In [ ]:
import pandas as pd
from src.preprocessing.load_dataset import load_dataset
from src.preprocessing.pipeline import preprocess
from src.preprocessing.transform_target import transform_target
from src.utils.save import save_processed, save_data_summary, save_splits
from src.preprocessing.split import split_data_train_val_test
from src.preprocessing.encode import encode_train_val_test
from src.preprocessing.scale import scale_train_val_test
from src.experiments.run_baseline import train_baseline
from src.experiments.run_traditional_fs import run_traditional_fs
from src.experiments.run_optimization_fs import run_optimization_fs
from src.experiments.run_xai_fs import run_xai_fs
from src.analysis.results_table import build_results_table
from src.visualization.plots import plot_auc_curves

## Define dataset name

In [ ]:
ds = "ibtracs.ALL.list.v04r01"

## Preprocessing

In [ ]:
df_raw, cfg = load_dataset(ds)
X_raw, y_raw = preprocess(df_raw.copy(), cfg)
y = transform_target(y_raw, cfg)
X = X_raw
save_processed(X, y, cfg)
save_data_summary(X, y, cfg)
splits = split_data_train_val_test(X, y)
# Encode: fit on train, transform val and test
splits["X_train"], splits["X_val"], splits["X_test"], encoder = encode_train_val_test(
    splits["X_train"],
    splits["X_val"],
    splits["X_test"]
)

# Scale: fit on train, transform val and test
splits["X_train"], splits["X_val"], splits["X_test"], scaler = scale_train_val_test(
    splits["X_train"],
    splits["X_val"],
    splits["X_test"]
)

save_splits(splits, cfg, encoder=encoder, scaler=scaler)

## Train baseline

In [ ]:
results = train_baseline(ds)

## Train with Traditional Feature Selection

In [ ]:
results = run_traditional_fs(ds)

## Train with Optimization based Feature Selection

In [ ]:
results = run_optimization_fs(ds)

## Train with XAI based Feature Selection

In [ ]:
results = run_xai_fs(ds)

## Analysis

In [ ]:
report, best, pivot_auc, pivot_f1 = build_results_table(ds)
plot_auc_curves(ds)